# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HemapavaniDontula/flyrank-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### 1. Ranked actions + reason codes

I create a ranked review queue using the available investigation signals rather than treating the model as an automatic decision maker.

Each content item receives a priority score and one or more reason codes. The reason codes explain which observable signals contributed to the recommendation.

The main reason codes are:

* `STALE_CONTENT` — the content has a relatively high `days_since_last_update`.
* `LOW_CTR` — CTR is below the median CTR for its `position_tier`.
* `HIGH_DECLINE_SIGNAL` — the model assigns a relatively high decline-severity score.
* `HIGH_POSITION` — the content is in `top_3` or `page_1`, so a change may deserve human review because the item already has a relatively strong observed position.

The queue is intended to prioritize investigation. A high priority does not mean that the content definitely needs to be changed.


In [2]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr

RANDOM_STATE = 42

# Load repository
repo_path = Path("/content/flyrank-ml")

if not repo_path.exists():
    !git clone --depth 1 https://github.com/HemapavaniDontula/flyrank-ml.git /content/flyrank-ml

os.chdir(repo_path)

# Load dataset
data_path = repo_path / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print("Dataset:", df.shape)

# Grouped split by client
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, val_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()

# Target
train_df["decline_severity"] = -pd.to_numeric(
    train_df["trend_pct"],
    errors="coerce"
)

val_df["decline_severity"] = -pd.to_numeric(
    val_df["trend_pct"],
    errors="coerce"
)

train_df = train_df.dropna(
    subset=["decline_severity"]
).copy()

val_df = val_df.dropna(
    subset=["decline_severity"]
).copy()

# Same feature set used in ML-08
feature_cols = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "position_tier",
    "avg_position",
    "ctr"
]

X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()

y_train = train_df["decline_severity"].copy()
y_val = val_df["decline_severity"].copy()

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "position_tier"
]

numeric_features = [
    col for col in feature_cols
    if col not in categorical_features
]

# Random Forest pipeline
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# Train model
rf_pipeline.fit(X_train, y_train)

# Generate validation predictions
model_pred = rf_pipeline.predict(X_val)

# Baseline
baseline_train = train_df.copy()
baseline_val = val_df.copy()

stale_cutoff = baseline_train[
    "days_since_last_update"
].quantile(2/3)

position_ctr_median = (
    baseline_train
    .groupby("position_tier")["ctr"]
    .median()
)

baseline_val["_stale_signal"] = (
    baseline_val["days_since_last_update"] >= stale_cutoff
).astype(int)

baseline_val["_position_ctr_median"] = (
    baseline_val["position_tier"]
    .map(position_ctr_median)
)

baseline_val["_ctr_signal"] = (
    baseline_val["ctr"] <
    baseline_val["_position_ctr_median"]
).astype(int)

baseline_val["_position_signal"] = (
    baseline_val["position_tier"].isin(
        ["top_3", "page_1"]
    )
).astype(int)

baseline_val["baseline_score"] = (
    2 * baseline_val["_stale_signal"]
    + 2 * baseline_val["_ctr_signal"]
    + baseline_val["_position_signal"]
)

# Ranking metrics
baseline_spearman = spearmanr(
    baseline_val["baseline_score"],
    y_val
).statistic

model_spearman = spearmanr(
    model_pred,
    y_val
).statistic

print("Setup completed successfully.")
print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Model predictions:", len(model_pred))
print("Model Spearman:", round(model_spearman, 4))
print("Baseline Spearman:", round(baseline_spearman, 4))

Cloning into '/content/flyrank-ml'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 91 (delta 9), reused 62 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 1.90 MiB | 6.44 MiB/s, done.
Resolving deltas: 100% (9/9), done.
Dataset: (30000, 44)
Setup completed successfully.
Train rows: 21133
Validation rows: 5479
Model predictions: 5479
Model Spearman: 0.1248
Baseline Spearman: 0.187


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create a human-review action queue

import pandas as pd
import numpy as np

action_queue = val_df[
    [
        "content_id",
        "client_id",
        "content_type",
        "position_tier",
        "days_since_last_update",
        "ctr",
        "trend_pct"
    ]
].copy()

# Add model prediction
action_queue["model_decline_score"] = model_pred

# Learn CTR median by position tier from training data only
train_position_ctr_median = (
    train_df
    .groupby("position_tier")["ctr"]
    .median()
)

# Map validation rows to their training-derived median
action_queue["position_ctr_median"] = (
    action_queue["position_tier"]
    .map(train_position_ctr_median)
)

# Reason-code signals
action_queue["STALE_CONTENT"] = (
    action_queue["days_since_last_update"] >=
    train_df["days_since_last_update"].quantile(2/3)
)

action_queue["LOW_CTR"] = (
    action_queue["ctr"] <
    action_queue["position_ctr_median"]
)

action_queue["HIGH_POSITION"] = (
    action_queue["position_tier"].isin(["top_3", "page_1"])
)

# Percentile-based model signal
model_threshold = action_queue["model_decline_score"].quantile(0.75)

action_queue["HIGH_DECLINE_SIGNAL"] = (
    action_queue["model_decline_score"] >= model_threshold
)

# Priority score
action_queue["priority_score"] = (
    action_queue["STALE_CONTENT"].astype(int) * 2
    + action_queue["LOW_CTR"].astype(int) * 2
    + action_queue["HIGH_DECLINE_SIGNAL"].astype(int) * 2
    + action_queue["HIGH_POSITION"].astype(int)
)

# Build human-readable reason codes
def make_reason_codes(row):
    reasons = []

    if row["STALE_CONTENT"]:
        reasons.append("STALE_CONTENT")

    if row["LOW_CTR"]:
        reasons.append("LOW_CTR")

    if row["HIGH_DECLINE_SIGNAL"]:
        reasons.append("HIGH_DECLINE_SIGNAL")

    if row["HIGH_POSITION"]:
        reasons.append("HIGH_POSITION")

    return ", ".join(reasons) if reasons else "NO_STRONG_SIGNAL"

action_queue["reason_codes"] = action_queue.apply(
    make_reason_codes,
    axis=1
)

# Highest-priority items first
action_queue = action_queue.sort_values(
    ["priority_score", "model_decline_score"],
    ascending=[False, False]
).reset_index(drop=True)

print("Action queue created.")
print("Rows:", len(action_queue))

print("\nTop 20 review candidates:")
print(
    action_queue[
        [
            "content_id",
            "priority_score",
            "reason_codes",
            "model_decline_score",
            "trend_pct"
        ]
    ].head(20).to_string(index=False)
)

Action queue created.
Rows: 5479

Top 20 review candidates:
          content_id  priority_score                                               reason_codes  model_decline_score  trend_pct
content_c6cffac60462               7 STALE_CONTENT, LOW_CTR, HIGH_DECLINE_SIGNAL, HIGH_POSITION            78.972500     -100.0
content_a159d40b38cd               7 STALE_CONTENT, LOW_CTR, HIGH_DECLINE_SIGNAL, HIGH_POSITION            76.474000     -100.0
content_4fc70e470460               7 STALE_CONTENT, LOW_CTR, HIGH_DECLINE_SIGNAL, HIGH_POSITION            72.707500      -62.1
content_f14a24375cc4               7 STALE_CONTENT, LOW_CTR, HIGH_DECLINE_SIGNAL, HIGH_POSITION            71.270375     -100.0
content_f55fd2d8ed04               7 STALE_CONTENT, LOW_CTR, HIGH_DECLINE_SIGNAL, HIGH_POSITION            68.666000      -84.9
content_2ba626fea4d6               7 STALE_CONTENT, LOW_CTR, HIGH_DECLINE_SIGNAL, HIGH_POSITION            67.094500       48.3
content_7766ffacdcfa               7 STALE_C

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### 2. Intended use and limits

The action queue is intended for content, SEO, or analytics teams that need to prioritize which content items deserve further investigation.

The queue combines observed content/search signals with the Random Forest model output to provide directional decision-support.

A high-priority item should be reviewed by a human before any action is taken.

The system is not intended to automatically change content, search strategy, metadata, or ranking-related decisions.

The results are limited by the available dataset, the grouped validation design, the selected features, and the observed outcome period. The model should not be assumed to generalize to new environments without further validation.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("INTENDED USE")
print("=" * 40)

print("Purpose: prioritize content for human review.")
print("Output: ranked action queue with reason codes.")
print("Decision type: directional decision-support.")

print("\nLimits:")
print("- Not an automatic content-change system.")
print("- Not causal evidence.")
print("- Not a guarantee of future decline.")
print("- Requires human review before action.")

INTENDED USE
Purpose: prioritize content for human review.
Output: ranked action queue with reason codes.
Decision type: directional decision-support.

Limits:
- Not an automatic content-change system.
- Not causal evidence.
- Not a guarantee of future decline.
- Requires human review before action.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### 3. Human review + the no-go list

Before acting on a recommendation, a human reviewer should check the content context, recent updates, CTR and position signals, and whether there are other business or editorial reasons that explain the observed pattern.

The reviewer should also verify that the recommended action is appropriate for the content type and intended audience.

The following actions should never be automated based only on this model:

* Automatically deleting or rewriting content.
* Automatically changing important business or editorial information.
* Automatically changing search strategy or ranking-related settings.
* Automatically treating a model score as proof of a ranking cause.
* Automatically applying the same recommendation to every client or content type.

The model provides a prioritization signal; the final decision remains with a human reviewer.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("HUMAN REVIEW CHECKLIST")
print("=" * 40)

review_checks = [
    "Check the actual content and its purpose.",
    "Check recent content updates.",
    "Review CTR and position context.",
    "Check whether the reason codes make sense.",
    "Check for business/editorial constraints.",
    "Confirm the proposed action is appropriate."
]

for i, check in enumerate(review_checks, 1):
    print(f"{i}. {check}")

print("\nNO-GO LIST")
print("=" * 40)

no_go = [
    "Automatic content deletion",
    "Automatic content rewriting",
    "Automatic ranking changes",
    "Treating model scores as causal evidence",
    "Applying recommendations without human review"
]

for item in no_go:
    print("-", item)

HUMAN REVIEW CHECKLIST
1. Check the actual content and its purpose.
2. Check recent content updates.
3. Review CTR and position context.
4. Check whether the reason codes make sense.
5. Check for business/editorial constraints.
6. Confirm the proposed action is appropriate.

NO-GO LIST
- Automatic content deletion
- Automatic content rewriting
- Automatic ranking changes
- Treating model scores as causal evidence
- Applying recommendations without human review


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### 4. Monitoring / retrain triggers

The recommendations may become stale if the underlying content population, search behavior, or relationship between the available signals and observed decline changes.

I would monitor the validation and operational signals over time.

Retraining or re-evaluation should be considered when:

* Model ranking performance decreases materially compared with the current baseline.
* The distribution of important input features changes substantially.
* Missingness patterns change.
* New content types or substantially different content populations appear.
* The relationship between model scores and observed decline severity changes.
* The action queue produces consistently poor or unhelpful recommendations during human review.

These triggers are intended as monitoring signals rather than fixed guarantees that retraining is required at a particular threshold.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("MONITORING / RETRAIN TRIGGERS")
print("=" * 40)

triggers = [
    "Decline in model ranking performance",
    "Major feature-distribution shift",
    "Changed missingness patterns",
    "New or substantially different content types",
    "Changed relationship between predictions and observed outcomes",
    "Repeatedly poor human-review outcomes"
]

for i, trigger in enumerate(triggers, 1):
    print(f"{i}. {trigger}")

print("\nCurrent validation Spearman:")
print(f"Random Forest: {model_spearman:.4f}")
print(f"Week-4 baseline: {baseline_spearman:.4f}")

MONITORING / RETRAIN TRIGGERS
1. Decline in model ranking performance
2. Major feature-distribution shift
3. Changed missingness patterns
4. New or substantially different content types
5. Changed relationship between predictions and observed outcomes
6. Repeatedly poor human-review outcomes

Current validation Spearman:
Random Forest: 0.1248
Week-4 baseline: 0.1870


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### 5. Exports for the paper

The ranked action queue is exported as a CSV file under `work/outputs/`.

The exported queue contains the content identifier, priority score, model signal, observed trend, and human-readable reason codes. These outputs can be reused in the paper as evidence of the decision-support workflow.

The exported file is an analysis artifact and should not be interpreted as an automated list of required content changes.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

output_dir = repo_path / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

queue_path = output_dir / "content_action_queue.csv"

export_columns = [
    "content_id",
    "client_id",
    "content_type",
    "position_tier",
    "days_since_last_update",
    "ctr",
    "trend_pct",
    "model_decline_score",
    "priority_score",
    "reason_codes"
]

action_queue[export_columns].to_csv(
    queue_path,
    index=False
)

print("Export completed.")
print("File:", queue_path)
print("Rows exported:", len(action_queue))

Export completed.
File: /content/flyrank-ml/work/outputs/content_action_queue.csv
Rows exported: 5479


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.